# Jaccard Similarity, Shingling, and Document Representation

## Learning Objectives
- Understand why document similarity is a fundamental problem in large-scale data mining
- Build shingle sets (character and word k-grams) from raw text
- Compute exact Jaccard similarity and understand its row-type decomposition
- Recognize why exact all-pairs similarity does not scale to millions of documents
- Represent documents as integer sets via shingle hashing and verify consistency with set Jaccard

## The Document Similarity Problem

Finding documents that are near-duplicates of each other is required in several important applications:

- **Plagiarism detection**: student assignments, academic papers, news articles copied from other sources
- **Web crawl deduplication**: large crawls collect the same page under many URLs; deduplication reduces index size and improves relevance
- **Entity resolution**: customer records, product listings, or biographic entries that describe the same real-world entity but differ in wording

The core challenge is *scale*: a web crawl may contain billions of documents. Reading every document pair is infeasible. We need a representation that is (a) compact, (b) preserves similarity, and (c) supports fast approximate search.

The pipeline is:

```
raw text  ->  shingles (sets)  ->  MinHash signatures (vectors)  ->  LSH buckets (candidates)
```

This notebook covers the first step: building and comparing shingle sets.

## k-Shingles (k-Grams)

A **k-shingle** (also called a k-gram) is a contiguous subsequence of length k drawn from a document.

**Character shingles** — extracted from the raw character stream:

```
text = "abcde",  k = 3
shingles = { "abc", "bcd", "cde" }
```

**Word shingles** — extracted from the token stream after splitting on whitespace:

```
text = "the cat sat on the mat",  k = 2
shingles = { ("the","cat"), ("cat","sat"), ("sat","on"), ("on","the"), ("the","mat") }
```

### Choosing k

| Use case | Recommended k |
|---|---|
| Web pages (character shingles) | 9 |
| Short text, tweets (character shingles) | 5 |
| Articles (word shingles) | 2 |
| DNA / code (character shingles) | 5-8 |

A larger k reduces the chance that two random documents share a shingle by coincidence, giving a more precise similarity signal. A smaller k is needed when documents are short (otherwise the shingle set would be nearly empty).

In [1]:
import re
from itertools import combinations

def char_shingles(text, k):
    # normalize: lowercase, collapse whitespace
    text = re.sub(r'\s+', ' ', text.lower().strip())
    return set(text[i:i+k] for i in range(len(text) - k + 1))

def word_shingles(text, k):
    # normalize: lowercase, strip punctuation, split
    text = re.sub(r'[^\w\s]', '', text.lower())
    tokens = text.split()
    return set(tuple(tokens[i:i+k]) for i in range(len(tokens) - k + 1))

# --- demo documents ---
docs = {
    "A": "The quick brown fox jumps over the lazy dog",
    # B is a near-duplicate of A (one word changed)
    "B": "The quick brown fox leaps over the lazy dog",
    # C has the same words in different order
    "C": "The lazy dog jumps over the quick brown fox",
    # D is topically related but differently worded
    "D": "A fast auburn fox hops across a sleepy hound",
    # E is unrelated
    "E": "Machine learning models learn from large datasets",
}

print('=== Character shingles (k=4) ===')
char_sets = {name: char_shingles(text, 4) for name, text in docs.items()}
for name, s in char_sets.items():
    print(f'  {name}: {len(s)} shingles  sample: {sorted(s)[:5]}')

print()
print('=== Word shingles (k=2) ===')
word_sets = {name: word_shingles(text, 2) for name, text in docs.items()}
for name, s in word_sets.items():
    print(f'  {name}: {len(s)} shingles  sample: {sorted(s)[:3]}')

=== Character shingles (k=4) ===
  A: 39 shingles  sample: [' bro', ' dog', ' fox', ' jum', ' laz']
  B: 39 shingles  sample: [' bro', ' dog', ' fox', ' laz', ' lea']
  C: 39 shingles  sample: [' bro', ' dog', ' fox', ' jum', ' laz']
  D: 41 shingles  sample: [' a s', ' acr', ' aub', ' fas', ' fox']
  E: 43 shingles  sample: [' dat', ' fro', ' lar', ' lea', ' mod']

=== Word shingles (k=2) ===
  A: 8 shingles  sample: [('brown', 'fox'), ('fox', 'jumps'), ('jumps', 'over')]
  B: 8 shingles  sample: [('brown', 'fox'), ('fox', 'leaps'), ('lazy', 'dog')]
  C: 8 shingles  sample: [('brown', 'fox'), ('dog', 'jumps'), ('jumps', 'over')]
  D: 8 shingles  sample: [('a', 'fast'), ('a', 'sleepy'), ('across', 'a')]
  E: 6 shingles  sample: [('from', 'large'), ('large', 'datasets'), ('learn', 'from')]


## Exact Jaccard Similarity

For two sets A and B the Jaccard similarity is:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

It ranges from 0 (disjoint sets) to 1 (identical sets).

### Row-Type Decomposition

Every possible shingle falls into one of four *row types*:

| Type | In A? | In B? | Count |
|------|-------|-------|-------|
| a | yes | yes | |A cap B| |
| b | yes | no  | |A \ B| |
| c | no  | yes | |B \ A| |
| d | no  | no  | (ignored) |

The Jaccard can then be written as:

$$J(A, B) = \frac{a}{a + b + c}$$

Type d rows (shingles in neither document) do not appear and are not counted — this is one of Jaccard's important properties for sparse high-dimensional data.

In [2]:
def jaccard(a, b):
    # direct set formula
    inter = len(a & b)
    union = len(a | b)
    return inter / union if union > 0 else 0.0

def jaccard_abcd(a, b):
    # explicit row-type counts
    a_only = len(a - b)       # b type
    b_only = len(b - a)       # c type
    both   = len(a & b)       # a type
    j = both / (both + a_only + b_only) if (both + a_only + b_only) > 0 else 0.0
    return j, both, a_only, b_only

print('=== Pairwise Jaccard (character shingles k=4) ===')
names = list(docs.keys())
print('    ', end='')
for n in names:
    print(f'  {n}   ', end='')
print()
for i in names:
    print(f'{i:4}', end='')
    for j in names:
        print(f'  {jaccard(char_sets[i], char_sets[j]):.3f}', end='')
    print()

print()
print('=== Selected pair detail (A vs B near-duplicate) ===')
j_val, a_cnt, b_cnt, c_cnt = jaccard_abcd(char_sets["A"], char_sets["B"])
print(f"  a (both)   = {a_cnt}")
print(f"  b (A only) = {b_cnt}")
print(f"  c (B only) = {c_cnt}")
print(f"  J = {a_cnt}/({a_cnt}+{b_cnt}+{c_cnt}) = {j_val:.4f}")

=== Pairwise Jaccard (character shingles k=4) ===
      A     B     C     D     E   
A     1.000  0.733  0.857  0.039  0.000
B     0.733  1.000  0.696  0.039  0.012
C     0.857  0.696  1.000  0.026  0.000
D     0.039  0.039  0.026  1.000  0.012
E     0.000  0.012  0.000  0.012  1.000

=== Selected pair detail (A vs B near-duplicate) ===
  a (both)   = 33
  b (A only) = 6
  c (B only) = 6
  J = 33/(33+6+6) = 0.7333


## The Scaling Problem

Given n documents, exact all-pairs comparison requires evaluating O(n^2) pairs. For each pair, computing the Jaccard costs O(|S|) where |S| is the average shingle-set size.

Total cost: O(n^2 * |S|)

| n (documents) | Pairs | Estimated time* |
|---|---|---|
| 10^3 | ~500 K | < 1 s |
| 10^6 | ~500 B | ~6 days |
| 10^9 | ~500 x 10^15 | > 15,000 years |

\* Assuming 10^8 set-intersection operations per second.

This motivates two key techniques:

1. **MinHash** (next notebook): compress each shingle set into a short integer signature of length k while *preserving* the Jaccard. Reduces per-document storage from |S| elements to k integers.
2. **LSH** (two notebooks forward): group documents into buckets so that only candidate pairs (likely to be similar) are evaluated, reducing the number of comparisons from O(n^2) to near-linear.

## Preprocessing Notes

Before shingling, normalize text to avoid spurious mismatches:

- **Lowercase**: 'The' and 'the' should be the same token
- **Strip punctuation**: 'dog.' vs 'dog' should not be different shingles
- **Collapse whitespace**: tabs, newlines, multiple spaces become a single space

When **not** to normalize:
- **Source code**: whitespace and case are semantically significant (Python indentation, C++ identifiers)
- **DNA sequences**: case carries no meaning but characters must be preserved exactly
- **LaTeX / Markdown**: punctuation is structural

### Hashing Shingles to Integers

A shingle set is a set of strings. Storing and comparing strings is slow. The standard practice is to map each shingle to a 32-bit or 64-bit integer using a hash function:

```python
shingle_id = hash(shingle) % (2**32)
```

This reduces memory and speeds up comparison. Two documents can then be represented as sets of integers, and the characteristic matrix has integer row indices.

In [3]:
import numpy as np

def shingle_to_int(shingle, bits=32):
    # Python's built-in hash is randomized per-process
    # use a stable polynomial hash for reproducibility
    h = 0
    for ch in str(shingle):
        h = (h * 31 + ord(ch)) & ((1 << bits) - 1)
    return h

# build integer shingle sets
int_sets = {
    name: {shingle_to_int(s) for s in char_shingles(text, 4)}
    for name, text in docs.items()
}

# build a small characteristic matrix
all_shingles = sorted(set.union(*int_sets.values()))
doc_names = list(docs.keys())
char_matrix = np.zeros((len(all_shingles), len(doc_names)), dtype=np.int8)
shingle_index = {s: i for i, s in enumerate(all_shingles)}
for col, name in enumerate(doc_names):
    for s in int_sets[name]:
        char_matrix[shingle_index[s], col] = 1

print(f'Characteristic matrix: {char_matrix.shape[0]} shingles x {char_matrix.shape[1]} documents')
print(f'Density: {char_matrix.mean():.3f} (fraction of 1s)')

print()
print('=== Verify: Jaccard from characteristic matrix vs set Jaccard ===')
for i, ni in enumerate(doc_names):
    for j, nj in enumerate(doc_names):
        if j <= i:
            continue
        col_i = char_matrix[:, i]
        col_j = char_matrix[:, j]
        a = int(np.sum((col_i == 1) & (col_j == 1)))
        b = int(np.sum((col_i == 1) & (col_j == 0)))
        c = int(np.sum((col_i == 0) & (col_j == 1)))
        j_matrix = a / (a + b + c) if (a + b + c) > 0 else 0.0
        j_set    = jaccard(int_sets[ni], int_sets[nj])
        match = 'OK' if abs(j_matrix - j_set) < 1e-9 else 'MISMATCH'
        print(f'  {ni}-{nj}: matrix={j_matrix:.4f}  set={j_set:.4f}  {match}')

Characteristic matrix: 127 shingles x 5 documents
Density: 0.317 (fraction of 1s)

=== Verify: Jaccard from characteristic matrix vs set Jaccard ===
  A-B: matrix=0.7333  set=0.7333  OK
  A-C: matrix=0.8571  set=0.8571  OK
  A-D: matrix=0.0390  set=0.0390  OK
  A-E: matrix=0.0000  set=0.0000  OK
  B-C: matrix=0.6957  set=0.6957  OK
  B-D: matrix=0.0390  set=0.0390  OK
  B-E: matrix=0.0123  set=0.0123  OK
  C-D: matrix=0.0256  set=0.0256  OK
  C-E: matrix=0.0000  set=0.0000  OK
  D-E: matrix=0.0120  set=0.0120  OK
